# PTHBV Data Download & Exploration

## Overview

This notebook downloads and explores **PTHBV (Precipitation & Temperature, Hydrological Agency's Water Model)** gridded data from SMHI. PTHBV is a climate database with a focus on hydrological model calculations.

### PTHBV Specifications

| Attribute | Value |
|-----------|-------|
| **Resolution** | 4 x 4 km |
| **Temporal** | Daily |
| **Coverage** | Sweden |
| **Period** | 1961-01-01 - ongoing |
| **Format** | NetCDF, CSV, JSON |
| **Parameters** | Air Temperature, Precipitation Amount |

### API Information

- **API Documentation**: https://opendata.smhi.se/pthbv/api
- **Note**: The PTHBV API endpoint serves a Docusaurus SPA. The actual data
  endpoints need to be discovered programmatically (this notebook does that).

### Notebook Goals

1. Discover the working PTHBV API endpoints
2. Download sample data programmatically
3. Parse NetCDF files and explore parameters (Temperature & Precipitation)
4. Visualize spatial and temporal coverage
5. Assess data quality for snow monitoring research

---
# Part 1: Setup and Dependencies

In [ ]:
# Install required packages if not already installed
# Uncomment and run if needed:

# !pip install xarray netCDF4 h5netcdf
# !pip install cartopy  # for map visualization

In [ ]:
# Import libraries
import os
import sys
import json
import requests
import time
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Basic libraries loaded!")

In [ ]:
# Try to import NetCDF handling libraries
NETCDF_AVAILABLE = False
try:
    import xarray as xr
    print(f"xarray version: {xr.__version__}")
    NETCDF_AVAILABLE = True
except ImportError as e:
    print(f"xarray not available: {e}")
    print("Run: pip install xarray netCDF4")

# Try cartopy for maps
CARTOPY_AVAILABLE = False
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_AVAILABLE = True
    print("cartopy available for map visualization")
except ImportError:
    print("cartopy not available - maps will be simple plots")
    print("Run: pip install cartopy (optional)")

In [ ]:
# Configuration
API_BASE = "https://opendata.smhi.se/pthbv/api"

# Output directory for PTHBV data
OUTPUT_DIR = Path('../data/raw/pthbv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"API Base: {API_BASE}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

---
# Part 2: API Discovery

The PTHBV API documentation page (`opendata.smhi.se/pthbv/api`) serves a
Docusaurus single-page application, so a simple `requests.get().json()` will
**not** work -- it returns HTML, not JSON.

We systematically probe candidate endpoint patterns to find the ones that
return real data.

In [ ]:
def probe_url(url: str, timeout: int = 15) -> dict:
    """
    Probe a URL and return status, content-type, size, and whether
    the body looks like JSON, XML, NetCDF/binary, or HTML.
    """
    try:
        resp = requests.get(url, timeout=timeout, stream=True)
        ct = resp.headers.get('content-type', '')
        # Read first 2 KB to classify
        head = resp.content[:2048]
        size = len(resp.content)
        
        if b'<!doctype' in head.lower() or b'<html' in head.lower():
            kind = 'HTML'
        elif head.startswith(b'{') or head.startswith(b'['):
            kind = 'JSON'
        elif head.startswith(b'<?xml') or b'<feed' in head[:200]:
            kind = 'XML'
        elif b'\x89HDF' in head[:8] or b'CDF' in head[:4]:
            kind = 'NetCDF'
        elif size > 5000 and b'<!doctype' not in head.lower():
            kind = 'BINARY'
        else:
            kind = 'OTHER'
        
        return {
            'url': url,
            'status': resp.status_code,
            'content_type': ct,
            'size': size,
            'kind': kind,
            'response': resp,
        }
    except Exception as e:
        return {
            'url': url,
            'status': -1,
            'content_type': '',
            'size': 0,
            'kind': 'ERROR',
            'error': str(e),
            'response': None,
        }


print("Probe helper defined!")

In [ ]:
# Systematic endpoint discovery
print("=" * 80)
print("PTHBV API ENDPOINT DISCOVERY")
print("=" * 80)

# Candidate URL patterns to test
candidates = [
    # Pattern 1: opendata.smhi.se/pthbv/... (documented base)
    f"{API_BASE}",
    f"{API_BASE}/version/1",
    f"{API_BASE}/version/1.json",
    f"{API_BASE}/version/1/parameter",
    f"{API_BASE}/version/1/parameter.json",
    f"{API_BASE}/version/1/parameter/tma",
    f"{API_BASE}/version/1/parameter/tma.json",
    f"{API_BASE}/version/1/parameter/psum",
    f"{API_BASE}/version/1/parameter/psum.json",
    # Pattern 2: point queries
    f"{API_BASE}/version/1/parameter/tma/point/59.33/18.07/data.json",
    f"{API_BASE}/version/1/parameter/tma/point/59.33/18.07/data.csv",
    f"{API_BASE}/version/1/parameter/tma/point/59.33;18.07/data.json",
    # Pattern 3: grid/netcdf
    f"{API_BASE}/version/1/parameter/tma/grid/data.nc",
    f"{API_BASE}/version/1/parameter/tma/data.nc",
    # Pattern 4: without /api/ prefix
    "https://opendata.smhi.se/pthbv/version/1.json",
    "https://opendata.smhi.se/pthbv/version/1/parameter.json",
    "https://opendata.smhi.se/pthbv/version/1/parameter/tma.json",
    # Pattern 5: alternative base URLs
    "https://opendata-download-grid-climate.smhi.se/api/version/latest.json",
    "https://opendata-download-metanalys.smhi.se/api/version/latest.json",
    # Pattern 6: grid archive feeds (MESAN is feed/6, try nearby IDs)
    "https://opendata-download-grid-archive.smhi.se/feed/1",
    "https://opendata-download-grid-archive.smhi.se/feed/2",
    "https://opendata-download-grid-archive.smhi.se/feed/3",
    "https://opendata-download-grid-archive.smhi.se/feed/4",
]

results = []
for url in candidates:
    r = probe_url(url)
    results.append(r)
    status_icon = {
        'JSON': '[JSON]', 'XML': '[XML]', 'NetCDF': '[NC]',
        'BINARY': '[BIN]', 'HTML': '[HTML]', 'ERROR': '[ERR]', 'OTHER': '[???]'
    }.get(r['kind'], '[???]')
    print(f"  {r['status']:3d} {status_icon:7s} {r['size']:>8,} B  {url}")

# Highlight non-HTML results
hits = [r for r in results if r['kind'] not in ('HTML', 'ERROR', 'OTHER')]
print(f"\nEndpoints returning data (non-HTML): {len(hits)}")
for h in hits:
    print(f"  [{h['kind']}] {h['url']}")

In [ ]:
# Show content of any JSON or XML responses found
print("=" * 80)
print("CONTENT OF DISCOVERED ENDPOINTS")
print("=" * 80)

for h in hits:
    print(f"\n--- {h['kind']}: {h['url']} ---")
    if h['kind'] == 'JSON':
        try:
            data = h['response'].json()
            print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])
        except:
            print(h['response'].text[:2000])
    elif h['kind'] == 'XML':
        print(h['response'].text[:3000])
    else:
        print(f"  Binary data, {h['size']:,} bytes")
        print(f"  Content-Type: {h['content_type']}")

if not hits:
    print("\nNo non-HTML endpoints found in the first probe.")
    print("The PTHBV API may require a different discovery approach.")
    print("See Part 3 for manual download alternatives.")

In [ ]:
# Extended probe: try date-specific download URLs
print("=" * 80)
print("PROBING DATE-SPECIFIC DOWNLOAD PATTERNS")
print("=" * 80)

date_candidates = [
    # Various guesses at how the daily download URL might look
    f"{API_BASE}/version/1/parameter/tma/data.nc?from=2023-01-01&to=2023-01-01",
    f"{API_BASE}/version/1/parameter/tma/2023/01/01/data.nc",
    f"{API_BASE}/version/1/parameter/tma/year/2023/month/01/day/01/data.nc",
    f"{API_BASE}/version/1/parameter/tma/date/2023-01-01.nc",
    f"{API_BASE}/version/1/parameter/tma/date/2023-01-01.json",
    f"{API_BASE}/version/1/parameter/tma/date/2023-01-01.csv",
    "https://opendata.smhi.se/pthbv/version/1/parameter/tma/data.nc?from=2023-01-01&to=2023-01-01",
    "https://opendata.smhi.se/pthbv/version/1/parameter/tma/2023/01/01/data.nc",
    # Try with Accept header for netcdf
    f"{API_BASE}/version/1/parameter/tma/point/59.33/18.07/data.netcdf",
]

date_results = []
for url in date_candidates:
    r = probe_url(url)
    date_results.append(r)
    status_icon = {'JSON': '[JSON]', 'XML': '[XML]', 'NetCDF': '[NC]',
                   'BINARY': '[BIN]', 'HTML': '[HTML]', 'ERROR': '[ERR]',
                   'OTHER': '[???]'}.get(r['kind'], '[???]')
    print(f"  {r['status']:3d} {status_icon:7s} {r['size']:>8,} B  {url}")

date_hits = [r for r in date_results if r['kind'] not in ('HTML', 'ERROR', 'OTHER')]
if date_hits:
    print(f"\nData endpoints found: {len(date_hits)}")
    for h in date_hits:
        print(f"  [{h['kind']}] {h['url']}")
else:
    print("\nNo data endpoints found with date patterns either.")
    print("\nThe PTHBV API likely requires JavaScript rendering or has")
    print("undocumented endpoints. See Part 3 for the manual approach.")

In [ ]:
# Store all discovered working endpoints for use later
all_hits = hits + date_hits if 'date_hits' in dir() else hits
WORKING_ENDPOINTS = {h['kind']: h['url'] for h in all_hits} if all_hits else {}

print("=" * 80)
print("API DISCOVERY SUMMARY")
print("=" * 80)

if WORKING_ENDPOINTS:
    print(f"\nWorking endpoints found ({len(WORKING_ENDPOINTS)}):")
    for kind, url in WORKING_ENDPOINTS.items():
        print(f"  {kind}: {url}")
else:
    print("""
No programmatic PTHBV data endpoints discovered.

This is expected -- the PTHBV API at opendata.smhi.se/pthbv/api is a
JavaScript single-page application (Docusaurus). All probed URL patterns
return the same SPA HTML shell regardless of path.

WORKAROUND OPTIONS:

1. MANUAL DOWNLOAD from the SMHI web interface:
   https://www.smhi.se/data/meteorologi/nederbord-och-temperatur/
   griddade-nederbords-och-temperaturdata
   -> Select parameter, date range, format (NetCDF)
   -> Place downloaded .nc files in: {output_dir}

2. Use the smhi-open-data Python package (if it supports PTHBV):
   pip install smhi-open-data

3. Contact SMHI for bulk data access.

This notebook will continue with analysis of any .nc files already
present in the output directory.
""".format(output_dir=OUTPUT_DIR.absolute()))

---
# Part 3: Download PTHBV Data

If programmatic endpoints were discovered above, we use them here.
Otherwise, this section provides the manual download instructions and
a download function ready to use once the correct URL pattern is known.

In [ ]:
def download_pthbv_file(url: str, output_dir: Path,
                        filename: str = None,
                        overwrite: bool = False) -> Optional[Path]:
    """
    Download a PTHBV file from a given URL.

    Args:
        url: Direct download URL
        output_dir: Directory to save file
        filename: Output filename (auto-detected from URL if None)
        overwrite: Whether to overwrite existing files

    Returns:
        Path to downloaded file, or None if failed
    """
    if filename is None:
        filename = url.split('/')[-1].split('?')[0]
        if not filename or filename == 'data.nc':
            filename = f"pthbv_{int(time.time())}.nc"

    output_path = output_dir / filename

    if output_path.exists() and not overwrite:
        print(f"  Skipping (exists): {filename}")
        return output_path

    try:
        print(f"  Downloading: {url}")
        resp = requests.get(url, timeout=300, stream=True)
        resp.raise_for_status()

        # Verify we got binary data, not HTML
        first_chunk = next(resp.iter_content(chunk_size=256))
        if b'<!doctype' in first_chunk.lower() or b'<html' in first_chunk.lower():
            print(f"  ERROR: Received HTML instead of data. URL may be wrong.")
            return None

        with open(output_path, 'wb') as f:
            f.write(first_chunk)
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f"  Saved: {filename} ({size_mb:.2f} MB)")
        return output_path

    except requests.exceptions.RequestException as e:
        print(f"  Download error: {e}")
        return None


print("Download function defined!")

In [ ]:
# Attempt programmatic download if we found working endpoints
print("=" * 80)
print("DOWNLOADING PTHBV DATA")
print("=" * 80)

downloaded_files = []

if WORKING_ENDPOINTS:
    print("\nUsing discovered endpoints...")
    for kind, url in WORKING_ENDPOINTS.items():
        if kind in ('NetCDF', 'BINARY', 'JSON', 'XML'):
            path = download_pthbv_file(url, OUTPUT_DIR)
            if path:
                downloaded_files.append(path)
else:
    print("""
No programmatic endpoints found.

MANUAL DOWNLOAD INSTRUCTIONS:
-------------------------------
1. Go to: https://www.smhi.se/data/meteorologi/nederbord-och-temperatur/
          griddade-nederbords-och-temperaturdata

2. Select:
   - Parameter: Temperature (tma) or Precipitation (psum)
   - Period: e.g. 2023-01-01 to 2023-01-07
   - Format: NetCDF

3. Download and place .nc files in:
   {output_dir}

Then re-run the cells below to explore the data.
""".format(output_dir=OUTPUT_DIR.absolute()))

print(f"\nDownloaded files: {len(downloaded_files)}")

In [ ]:
# List whatever files we have (downloaded or manually placed)
print("=" * 80)
print("FILES IN OUTPUT DIRECTORY")
print("=" * 80)

pthbv_files = sorted(list(OUTPUT_DIR.glob('*.nc')))
other_files = sorted([f for f in OUTPUT_DIR.iterdir()
                       if f.is_file() and f.suffix != '.nc'])

print(f"\nNetCDF files: {len(pthbv_files)}")
for f in pthbv_files[:20]:
    print(f"  {f.name}: {f.stat().st_size / 1024 / 1024:.2f} MB")
if len(pthbv_files) > 20:
    print(f"  ... and {len(pthbv_files) - 20} more")

if other_files:
    print(f"\nOther files: {len(other_files)}")
    for f in other_files[:10]:
        print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

if not pthbv_files and not other_files:
    print("\nNo files found. Please download data manually (see instructions above).")

total_size = sum(f.stat().st_size for f in pthbv_files) / 1024 / 1024
print(f"\nTotal NetCDF size: {total_size:.2f} MB")

---
# Part 4: Explore Downloaded NetCDF Files

PTHBV data is in NetCDF format. The variable and coordinate names are
auto-detected below so the code works regardless of the exact naming
convention used by SMHI.

In [ ]:
def explore_netcdf(filepath: Path) -> dict:
    """
    Open a NetCDF file and print its full structure.
    Returns a dict with variable names, coords, dims, attrs.
    """
    if not NETCDF_AVAILABLE:
        print("xarray not available.")
        return {}

    ds = xr.open_dataset(filepath)

    info = {
        'filepath': str(filepath),
        'variables': list(ds.data_vars),
        'coordinates': list(ds.coords),
        'dimensions': dict(ds.dims),
        'attrs': dict(ds.attrs),
    }

    print(f"File: {filepath.name}")
    print(f"Size: {filepath.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"\nDimensions: {dict(ds.dims)}")
    print(f"Coordinates: {list(ds.coords)}")
    print(f"\nData variables ({len(ds.data_vars)}):")
    for var in ds.data_vars:
        v = ds[var]
        print(f"  {var:20s} shape={str(v.shape):25s} dtype={v.dtype}")
        print(f"  {'':20s} units={v.attrs.get('units', '?'):12s} "
              f"long_name={v.attrs.get('long_name', v.attrs.get('standard_name', '?'))}")
        print(f"  {'':20s} min={float(v.min()):.4g}, mean={float(v.mean()):.4g}, max={float(v.max()):.4g}")

    if ds.attrs:
        print(f"\nGlobal attributes:")
        for k, v in list(ds.attrs.items())[:10]:
            print(f"  {k}: {v}")

    ds.close()
    return info


print("NetCDF exploration function defined!")

In [ ]:
# Explore each unique file type
if pthbv_files and NETCDF_AVAILABLE:
    print("=" * 80)
    print("EXPLORING NETCDF FILES")
    print("=" * 80)

    file_infos = []
    # Explore up to 3 files to understand structure
    for f in pthbv_files[:3]:
        print(f"\n{'─' * 60}")
        info = explore_netcdf(f)
        file_infos.append(info)

    # Summary of discovered variable names
    all_vars = set()
    all_coords = set()
    for info in file_infos:
        all_vars.update(info.get('variables', []))
        all_coords.update(info.get('coordinates', []))

    print(f"\n{'=' * 60}")
    print(f"All variable names found: {sorted(all_vars)}")
    print(f"All coordinate names found: {sorted(all_coords)}")
else:
    file_infos = []
    all_vars = set()
    all_coords = set()
    print("No NetCDF files to explore.")

In [ ]:
# Auto-detect coordinate and variable names
# PTHBV files may use lat/lon, latitude/longitude, y/x, etc.

def detect_names(filepath: Path) -> dict:
    """
    Open a NetCDF file and detect the lat, lon, time, and data variable names.
    """
    ds = xr.open_dataset(filepath)

    names = {'lat': None, 'lon': None, 'time': None, 'data_vars': []}

    # Detect lat
    for candidate in ['lat', 'latitude', 'y', 'LAT', 'Latitude']:
        if candidate in ds.coords or candidate in ds.dims:
            names['lat'] = candidate
            break
    # Detect lon
    for candidate in ['lon', 'longitude', 'x', 'LON', 'Longitude']:
        if candidate in ds.coords or candidate in ds.dims:
            names['lon'] = candidate
            break
    # Detect time
    for candidate in ['time', 'Time', 'date', 'DATE']:
        if candidate in ds.coords or candidate in ds.dims:
            names['time'] = candidate
            break
    # Data variables
    names['data_vars'] = list(ds.data_vars)

    ds.close()
    return names


if pthbv_files and NETCDF_AVAILABLE:
    NAMES = detect_names(pthbv_files[0])
    print("Auto-detected names:")
    print(f"  Latitude coord:  {NAMES['lat']}")
    print(f"  Longitude coord: {NAMES['lon']}")
    print(f"  Time coord:      {NAMES['time']}")
    print(f"  Data variables:  {NAMES['data_vars']}")
else:
    # Defaults based on SMHI convention
    NAMES = {'lat': 'lat', 'lon': 'lon', 'time': 'time', 'data_vars': []}
    print("No files to detect names from. Using defaults.")

---
# Part 5: Spatial Visualization

Plot temperature and precipitation maps from the PTHBV data.

In [ ]:
def plot_pthbv_map(ds, var_name, title, cmap, units, norm=None,
                   lat_name=None, lon_name=None, figsize=(10, 12)):
    """
    Plot a 2D map of a PTHBV variable.
    Auto-detects lat/lon coordinate names if not provided.
    """
    lat_name = lat_name or NAMES['lat']
    lon_name = lon_name or NAMES['lon']

    if lat_name is None or lon_name is None:
        print("Cannot plot: lat/lon coordinates not detected.")
        return

    data = ds[var_name].squeeze()
    lats = ds[lat_name].values
    lons = ds[lon_name].values

    fig, ax = plt.subplots(figsize=figsize,
                           subplot_kw={'projection': ccrs.Mercator()} if CARTOPY_AVAILABLE else {})

    plot_kwargs = dict(cmap=cmap)
    if norm is not None:
        plot_kwargs['norm'] = norm

    if CARTOPY_AVAILABLE:
        if lats.ndim == 2:
            im = ax.pcolormesh(lons, lats, data.values, transform=ccrs.PlateCarree(), **plot_kwargs)
        else:
            im = data.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                                      x=lon_name, y=lat_name, add_colorbar=False, **plot_kwargs)
        ax.coastlines('50m', color='black', linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
        ax.set_extent([5, 25, 55, 70], crs=ccrs.PlateCarree())
        gl = ax.gridlines(draw_labels=True, alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
    else:
        if lats.ndim == 2:
            im = ax.pcolormesh(lons, lats, data.values, **plot_kwargs)
        else:
            im = data.plot.pcolormesh(ax=ax, x=lon_name, y=lat_name,
                                      add_colorbar=False, **plot_kwargs)

    cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.08)
    cbar.set_label(units)
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.show()


print("Map plotting function defined!")

In [ ]:
# Plot first available file
if pthbv_files and NETCDF_AVAILABLE and NAMES['data_vars']:
    ds = xr.open_dataset(pthbv_files[0])

    for var_name in NAMES['data_vars']:
        print(f"\nPlotting variable: {var_name}")
        data = ds[var_name].squeeze()
        units = ds[var_name].attrs.get('units', '?')
        long_name = ds[var_name].attrs.get('long_name',
                     ds[var_name].attrs.get('standard_name', var_name))

        # Determine time label
        time_label = ''
        if NAMES['time'] and NAMES['time'] in ds.coords:
            try:
                t = pd.to_datetime(ds[NAMES['time']].values[0])
                time_label = t.strftime('%Y-%m-%d')
            except:
                time_label = str(ds[NAMES['time']].values[0])

        # Choose colormap and normalization
        is_temp = any(k in var_name.lower() or k in long_name.lower()
                      for k in ['temp', 'tma', 'temperature'])
        is_precip = any(k in var_name.lower() or k in long_name.lower()
                        for k in ['prec', 'psum', 'precipitation', 'rain'])

        if is_temp:
            vals = data.values.flatten()
            vals = vals[~np.isnan(vals)]
            abs_max = max(abs(np.min(vals)), abs(np.max(vals)))
            norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
            cmap = 'RdBu_r'
        elif is_precip:
            norm = None
            cmap = 'Blues'
        else:
            norm = None
            cmap = 'viridis'

        title = f'PTHBV {long_name}'
        if time_label:
            title += f'\n{time_label}'
        title += f'\n{pthbv_files[0].name}'

        plot_pthbv_map(ds, var_name, title, cmap, units, norm=norm)

        # Print statistics
        vals = data.values.flatten()
        vals = vals[~np.isnan(vals)]
        print(f"  Stats: min={np.min(vals):.2f}, mean={np.mean(vals):.2f}, "
              f"max={np.max(vals):.2f}, NaN={np.sum(np.isnan(data.values))}")

    ds.close()
else:
    print("No data available for plotting.")

---
# Part 6: Time Series Analysis

Combine multiple daily files and extract time series at Nordic locations.

In [ ]:
# Nordic locations of interest
locations = {
    'Stockholm': (59.33, 18.07),
    'Kiruna': (67.86, 20.22),
    'Gothenburg': (57.71, 11.97),
    'Ostersund': (63.18, 14.64),
    'Lulea': (65.58, 22.15),
}

if len(pthbv_files) >= 2 and NETCDF_AVAILABLE:
    print("=" * 80)
    print("TIME SERIES ANALYSIS")
    print("=" * 80)

    # Open all files as a combined dataset
    try:
        ds_all = xr.open_mfdataset(pthbv_files, combine='by_coords')
        print(f"\nCombined dataset:")
        print(f"  Variables: {list(ds_all.data_vars)}")
        print(f"  Time range: {ds_all[NAMES['time']].values[0]} to {ds_all[NAMES['time']].values[-1]}")
        print(f"  Time steps: {len(ds_all[NAMES['time']])}")
    except Exception as e:
        print(f"Error combining files: {e}")
        print("Trying to open files individually...")
        ds_all = None

    if ds_all is not None:
        for var_name in NAMES['data_vars']:
            long_name = ds_all[var_name].attrs.get('long_name',
                         ds_all[var_name].attrs.get('standard_name', var_name))
            units = ds_all[var_name].attrs.get('units', '?')

            fig, ax = plt.subplots(figsize=(14, 6))

            for name, (lat, lon) in locations.items():
                try:
                    ts = ds_all[var_name].sel(
                        **{NAMES['lat']: lat, NAMES['lon']: lon},
                        method='nearest'
                    )
                    ax.plot(ts[NAMES['time']], ts.values, 'o-',
                            label=name, markersize=5, linewidth=1.5)
                except Exception as e:
                    print(f"  Could not extract {name}: {e}")

            ax.set_xlabel('Date')
            ax.set_ylabel(f'{long_name} ({units})')
            ax.set_title(f'PTHBV {long_name} at Nordic Locations')
            ax.legend()
            ax.grid(True, alpha=0.3)

            is_temp = any(k in var_name.lower() for k in ['temp', 'tma'])
            if is_temp:
                ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)

            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.show()

        ds_all.close()

elif len(pthbv_files) == 1 and NETCDF_AVAILABLE:
    print("Only 1 file available -- skipping time series (need >= 2 timesteps).")
else:
    print("No data available for time series analysis.")

In [ ]:
# Dual-axis temperature + precipitation plot for Stockholm
if len(pthbv_files) >= 2 and NETCDF_AVAILABLE and len(NAMES['data_vars']) >= 2:
    ds_all = xr.open_mfdataset(pthbv_files, combine='by_coords')

    lat_pt, lon_pt = 59.33, 18.07
    sel_kwargs = {NAMES['lat']: lat_pt, NAMES['lon']: lon_pt}

    # Find temp and precip variables
    temp_var = next((v for v in NAMES['data_vars']
                     if any(k in v.lower() for k in ['temp', 'tma'])), None)
    precip_var = next((v for v in NAMES['data_vars']
                       if any(k in v.lower() for k in ['prec', 'psum'])), None)

    if temp_var and precip_var:
        ts_temp = ds_all[temp_var].sel(**sel_kwargs, method='nearest')
        ts_precip = ds_all[precip_var].sel(**sel_kwargs, method='nearest')

        fig, ax1 = plt.subplots(figsize=(15, 7))

        color_t = 'tab:red'
        ax1.set_xlabel('Date')
        ax1.set_ylabel(f'{temp_var} ({ds_all[temp_var].attrs.get("units", "C")})', color=color_t)
        ax1.plot(ts_temp[NAMES['time']], ts_temp.values, color=color_t, marker='o', linestyle='-')
        ax1.tick_params(axis='y', labelcolor=color_t)
        ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
        ax1.grid(True, alpha=0.3)

        ax2 = ax1.twinx()
        color_p = 'tab:blue'
        ax2.set_ylabel(f'{precip_var} ({ds_all[precip_var].attrs.get("units", "mm")})', color=color_p)
        ax2.bar(ts_precip[NAMES['time']].values, ts_precip.values,
                color=color_p, alpha=0.6, width=0.5)
        ax2.tick_params(axis='y', labelcolor=color_p)

        actual_lat = float(ts_temp[NAMES['lat']])
        actual_lon = float(ts_temp[NAMES['lon']])
        plt.title(f'PTHBV Temperature & Precipitation near Stockholm\n'
                  f'(Nearest grid point: {actual_lat:.2f}N, {actual_lon:.2f}E)')
        fig.tight_layout()
        plt.show()

        ds_all.close()
    else:
        print(f"Could not identify temp and/or precip variables from: {NAMES['data_vars']}")
        ds_all.close()
else:
    print("Not enough data for dual-axis plot.")

---
# Part 7: Grid Structure & Data Quality

In [ ]:
# Grid structure analysis
if pthbv_files and NETCDF_AVAILABLE:
    print("=" * 80)
    print("PTHBV GRID STRUCTURE")
    print("=" * 80)

    ds = xr.open_dataset(pthbv_files[0])

    lat_name = NAMES['lat']
    lon_name = NAMES['lon']

    if lat_name and lon_name:
        lats = ds[lat_name].values
        lons = ds[lon_name].values

        print(f"\nLat coordinate: {lat_name}")
        print(f"  Shape: {lats.shape}")
        print(f"  Range: {lats.min():.4f} to {lats.max():.4f}")

        print(f"\nLon coordinate: {lon_name}")
        print(f"  Shape: {lons.shape}")
        print(f"  Range: {lons.min():.4f} to {lons.max():.4f}")

        if lats.ndim == 1:
            dlat = np.abs(np.diff(lats)).mean()
            dlon = np.abs(np.diff(lons)).mean()
            n_cells = len(lats) * len(lons)
            print(f"\nGrid type: Regular 1D")
            print(f"  Lat points: {len(lats)}")
            print(f"  Lon points: {len(lons)}")
            print(f"  Total grid cells: {n_cells:,}")
            print(f"  Lat spacing: {dlat:.4f} deg (~{dlat * 111:.1f} km)")
            print(f"  Lon spacing: {dlon:.4f} deg (~{dlon * 111 * np.cos(np.radians(63)):.1f} km at 63N)")
        elif lats.ndim == 2:
            n_cells = lats.shape[0] * lats.shape[1]
            print(f"\nGrid type: Curvilinear 2D")
            print(f"  Shape: {lats.shape}")
            print(f"  Total grid cells: {n_cells:,}")

    # Dimensions
    print(f"\nAll dimensions: {dict(ds.dims)}")

    ds.close()
else:
    print("No data available.")

In [ ]:
# Missing value / NaN check
if pthbv_files and NETCDF_AVAILABLE:
    print("=" * 80)
    print("DATA QUALITY: MISSING VALUES")
    print("=" * 80)

    ds = xr.open_dataset(pthbv_files[0])

    for var_name in NAMES['data_vars']:
        vals = ds[var_name].values
        total = vals.size
        n_nan = np.sum(np.isnan(vals))
        n_valid = total - n_nan
        pct = 100 * n_valid / total if total > 0 else 0
        print(f"\n  {var_name}:")
        print(f"    Total pixels: {total:,}")
        print(f"    Valid:        {n_valid:,} ({pct:.1f}%)")
        print(f"    NaN:          {n_nan:,} ({100 - pct:.1f}%)")

    ds.close()
else:
    print("No data available.")

In [ ]:
# Histograms of all variables
if pthbv_files and NETCDF_AVAILABLE:
    ds = xr.open_dataset(pthbv_files[0])
    n_vars = len(NAMES['data_vars'])

    if n_vars > 0:
        fig, axes = plt.subplots(1, n_vars, figsize=(6 * n_vars, 4))
        if n_vars == 1:
            axes = [axes]

        for idx, var_name in enumerate(NAMES['data_vars']):
            ax = axes[idx]
            vals = ds[var_name].values.flatten()
            vals = vals[~np.isnan(vals)]
            units = ds[var_name].attrs.get('units', '?')

            ax.hist(vals, bins=60, color=plt.cm.tab10(idx), alpha=0.75, edgecolor='white')
            ax.set_xlabel(f'{var_name} ({units})')
            ax.set_ylabel('Pixel Count')
            ax.set_title(f'{var_name} Distribution')
            ax.axvline(np.mean(vals), color='red', linestyle='--',
                       label=f'Mean={np.mean(vals):.1f}')
            ax.legend(fontsize=9)

        fig.suptitle(f'PTHBV Spatial Distributions\n{pthbv_files[0].name}', fontsize=13)
        plt.tight_layout()
        plt.show()

    ds.close()
else:
    print("No data available.")

---
# Part 8: Snow Monitoring Relevance

In [ ]:
print("=" * 80)
print("PTHBV - SNOW MONITORING RELEVANCE")
print("=" * 80)

print("""
PTHBV provides ONLY 2 parameters: Temperature and Precipitation.

Role in the snow monitoring project:

  Strength                          Limitation
  ────────────────────────────────  ──────────────────────────────────
  Daily data since 1961 (60+ yrs)  Only 2 parameters (no wind,
  4 km gridded, Sweden coverage     humidity, clouds, snow depth)
  Complete spatial coverage          Coarser than MESAN (4 vs 2.5 km)
  No missing values                  Daily only (no sub-daily)
  Precipitation wind-corrected       Sweden only (not Scandinavia)

RECOMMENDED USAGE:
  1. LONG-TERM CLIMATE BASELINE (1961-2014, before MESAN exists)
     - Multi-decadal temperature and precipitation trends
     - Historical snow season characterization
  2. COMPLEMENT TO MESAN (2015+)
     - Cross-validate MESAN temperature and precipitation
     - Provide independent second estimate
  3. SIMPLE SNOW PROXY
     - Precipitation on days with T < 0 C is likely snow
     - Rough snow accumulation / melt estimation

COMPARISON WITH MESAN:
  Feature           PTHBV              MESAN
  ──────────────    ─────────────────  ──────────────────
  Resolution        4 km               2.5 km
  Temporal          Daily              Hourly
  Period            1961-ongoing       2014-ongoing
  Parameters        2                  ~30
  Format            NetCDF/CSV/JSON    GRIB
  Coverage          Sweden             Scandinavia + N. Europe
""")

---
# Part 9: Summary & Next Steps

In [ ]:
print("=" * 80)
print("SUMMARY")
print("=" * 80)

nc_count = len(list(OUTPUT_DIR.glob('*.nc')))
total_size = sum(f.stat().st_size for f in OUTPUT_DIR.glob('*.nc')) / 1024 / 1024

print(f"""
Downloaded Data:
  NetCDF files: {nc_count}
  Total size: {total_size:.2f} MB
  Directory: {OUTPUT_DIR.absolute()}

PTHBV Dataset:
  Resolution: 4 x 4 km
  Temporal: Daily
  Period: 1961-01-01 to ongoing
  Parameters: Temperature (C), Precipitation (mm)
  Formats: NetCDF, CSV, JSON
  Coverage: Sweden

API Status:
  Working endpoints found: {len(WORKING_ENDPOINTS)}
  Note: If 0, use manual download from SMHI web interface.

Detected variable/coordinate names:
  Lat: {NAMES['lat']}
  Lon: {NAMES['lon']}
  Time: {NAMES['time']}
  Variables: {NAMES['data_vars']}
""")

In [ ]:
print("=" * 80)
print("NEXT STEPS")
print("=" * 80)

print("""
1. DOWNLOAD DATA (if not yet done):
   - Manual: https://www.smhi.se/data/meteorologi/nederbord-och-temperatur/
             griddade-nederbords-och-temperaturdata
   - Select temperature + precipitation, NetCDF format
   - Suggested periods:
     a) Recent winter: 2023-12-01 to 2024-03-31
     b) Multi-year: 2015-01-01 to 2024-12-31 (overlap with MESAN)
     c) Full history: 1961-01-01 to 2024-12-31 (climate baseline)

2. COMBINE WITH MESAN:
   - Regrid PTHBV (4 km) to MESAN grid (2.5 km) or vice versa
   - Cross-validate temperature and precipitation fields
   - Use PTHBV for pre-2015 training data

3. DERIVE SNOW PROXY:
   - Snow day = precipitation > 0 AND temperature < 0 C
   - Cumulative snowfall estimation
   - Compare with MESAN frozen precipitation variables

4. UPDATE MODEL:
   - Add PTHBV as auxiliary input for long-term context
   - Or use PTHBV for pre-training, fine-tune on MESAN
""")

In [ ]:
# Save exploration results
results_out = {
    'timestamp': datetime.now().isoformat(),
    'netcdf_files': nc_count,
    'total_size_mb': round(total_size, 2),
    'output_dir': str(OUTPUT_DIR.absolute()),
    'working_endpoints': WORKING_ENDPOINTS,
    'detected_names': {k: v for k, v in NAMES.items() if k != 'data_vars'},
    'detected_variables': NAMES.get('data_vars', []),
    'api_note': 'PTHBV API serves Docusaurus SPA; manual download may be needed',
}

results_path = OUTPUT_DIR / '_exploration_results.json'
with open(results_path, 'w') as f:
    json.dump(results_out, f, indent=2, default=str)

print(f"Exploration results saved to: {results_path}")

---
*Notebook created for Nordic Snow Monitoring Project*  
*RISE Research Institutes of Sweden*